In [1]:
from google.colab import files
import pandas as pd
up = files.upload()   # choose predictions_partial.csv
PATH = next(iter(up))
df = pd.read_csv(PATH)
print('rows:', len(df), '| cols:', [c for c in df.columns if 'pred' in c or 'score' in c])
print('Amharic rows:', (df['Language']=='Amharic').sum())
print('Has OriginalText:', 'OriginalText' in df.columns)

Saving predictions_partial.csv to predictions_partial (1).csv
rows: 838 | cols: ['persp_score', 'persp_pred', 'afrihate_pred']
Amharic rows: 470
Has OriginalText: True


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
MODEL = 'amengemeda/amharic-hate-speech-detection-mBERT'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()
if torch.cuda.is_available(): model = model.cuda()
print('id2label:', model.config.id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id2label: {0: 'LABEL_0', 1: 'LABEL_1'}


In [3]:
# sanity check: which label is "hate"?
tests = [
    "እነዚህን ሰዎች ማጥፋት አለብን",          # "we must eliminate these people" (hateful)
    "ዛሬ ጥሩ የአየር ሁኔታ ነው",              # "the weather is nice today" (neutral)
]
for s in tests:
    enc = tok(s, return_tensors='pt', truncation=True, max_length=128)
    if torch.cuda.is_available(): enc = {k:v.cuda() for k,v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred = int(logits.argmax(-1))
    print(f"pred={pred}  | {s}")

pred=1  | እነዚህን ሰዎች ማጥፋት አለብን
pred=0  | ዛሬ ጥሩ የአየር ሁኔታ ነው


In [4]:
hate_ids = {1}   # confirmed: label 1 = hate
MAXLEN = 128
labels, preds = [], []
amh_mask = df['Language'] == 'Amharic'
rows = df[amh_mask]
for n, (i, row) in enumerate(rows.iterrows()):
    enc = tok(str(row['OriginalText'])[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt')
    if torch.cuda.is_available(): enc = {k: v.cuda() for k, v in enc.items()}
    with torch.no_grad():
        idx = int(model(**enc).logits.argmax(-1))
    labels.append(idx); preds.append(int(idx in hate_ids))
    if (n+1) % 100 == 0: print(f'  mBERT {n+1}/{len(rows)}')
df['ammbert_label'] = None
df['ammbert_pred'] = None
df.loc[amh_mask, 'ammbert_label'] = labels
df.loc[amh_mask, 'ammbert_pred'] = preds
print('Amharic mBERT Hate predictions:', sum(preds), 'of', len(preds), 'Amharic rows')

  mBERT 100/470
  mBERT 200/470
  mBERT 300/470
  mBERT 400/470
Amharic mBERT Hate predictions: 213 of 470 Amharic rows


In [5]:
df.to_csv('predictions_full.csv', index=False)
from google.colab import files
files.download('predictions_full.csv')
print('Saved predictions_full.csv with:',
      [c for c in df.columns if 'pred' in c or 'score' in c])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved predictions_full.csv with: ['persp_score', 'persp_pred', 'afrihate_pred', 'ammbert_pred']


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd
if 'gold_bin' not in df.columns:
    df['gold_bin'] = (df['Label3Class'].astype(str).str.strip()=='Hate').astype(int)
CLF = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred'), ('Amharic mBERT','ammbert_pred')]
def mrow(name, scope, frame, col):
    s = frame[frame[col].notna()]
    if len(s)==0: return None
    yt=s['gold_bin'].astype(int); yp=s[col].astype(int)
    tn,fp,fn,tp = confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {'classifier':name,'scope':scope,
            'coverage':f'{len(s)}/{len(frame)} ({len(s)/len(frame):.0%})',
            'accuracy':round(accuracy_score(yt,yp),3),'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}
rows=[]
for name,col in CLF:
    if col not in df.columns: continue
    rows.append(mrow(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r=mrow(name,lang,df[df['Language']==lang],col)
        if r: rows.append(r)
res=pd.DataFrame([r for r in rows if r])
print(res.to_string(index=False))
res.to_csv('stage2_results_full.csv', index=False)
files.download('stage2_results_full.csv')

   classifier      scope       coverage  accuracy  precision  recall    f1  TP  FP  FN  TN
  Perspective    Overall  687/838 (82%)     0.345      1.000   0.104 0.188  52   0 450 185
  Perspective    Amharic  409/470 (87%)     0.347      1.000   0.113 0.203  34   0 267 108
  Perspective Afan Oromo  278/368 (76%)     0.342      1.000   0.090 0.164  18   0 183  77
     AfriHate    Overall 838/838 (100%)     0.567      0.918   0.430 0.586 257  23 340 218
     AfriHate    Amharic 470/470 (100%)     0.704      0.914   0.640 0.753 212  20 119 119
     AfriHate Afan Oromo 368/368 (100%)     0.391      0.938   0.169 0.287  45   3 221  99
Amharic mBERT    Overall  470/838 (56%)     0.626      0.864   0.556 0.676 184  29 147 110
Amharic mBERT    Amharic 470/470 (100%)     0.626      0.864   0.556 0.676 184  29 147 110


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd
if 'gold_bin' not in df.columns:
    df['gold_bin'] = (df['Label3Class'].astype(str).str.strip()=='Hate').astype(int)
CLF = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred'), ('Amharic mBERT','ammbert_pred')]
def mrow(name, scope, frame, col):
    s = frame[frame[col].notna()]
    if len(s)==0: return None
    yt=s['gold_bin'].astype(int); yp=s[col].astype(int)
    tn,fp,fn,tp = confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {'classifier':name,'scope':scope,
            'coverage':f'{len(s)}/{len(frame)} ({len(s)/len(frame):.0%})',
            'accuracy':round(accuracy_score(yt,yp),3),'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}
rows=[]
for name,col in CLF:
    if col not in df.columns: continue
    rows.append(mrow(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r=mrow(name,lang,df[df['Language']==lang],col)
        if r: rows.append(r)
res=pd.DataFrame([r for r in rows if r])
print(res.to_string(index=False))
res.to_csv('stage2_results_full.csv', index=False)
from google.colab import files
files.download('stage2_results_full.csv')

   classifier      scope       coverage  accuracy  precision  recall    f1  TP  FP  FN  TN
  Perspective    Overall  687/838 (82%)     0.345      1.000   0.104 0.188  52   0 450 185
  Perspective    Amharic  409/470 (87%)     0.347      1.000   0.113 0.203  34   0 267 108
  Perspective Afan Oromo  278/368 (76%)     0.342      1.000   0.090 0.164  18   0 183  77
     AfriHate    Overall 838/838 (100%)     0.567      0.918   0.430 0.586 257  23 340 218
     AfriHate    Amharic 470/470 (100%)     0.704      0.914   0.640 0.753 212  20 119 119
     AfriHate Afan Oromo 368/368 (100%)     0.391      0.938   0.169 0.287  45   3 221  99
Amharic mBERT    Overall  470/838 (56%)     0.626      0.864   0.556 0.676 184  29 147 110
Amharic mBERT    Amharic 470/470 (100%)     0.626      0.864   0.556 0.676 184  29 147 110


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>